In [18]:
import os
import joblib
import pandas as pd
import numpy as np

class ProteinRecommendationPipeline:
    def __init__(self):
        # مسیر پوشه مدل‌های شما
        self.models_dir = r"C:\Windows\System32\protein-recommendation-ai\protein-recommendation-ai\models"
        
        self.regressor_path = os.path.join(self.models_dir, 'random_forest_protein_model.pkl')
        self.classifier_path = os.path.join(self.models_dir, 'trained_supplement_classifier.pkl')
        
        if os.path.exists(self.regressor_path) and os.path.exists(self.classifier_path):
            self.regressor = joblib.load(self.regressor_path)
            self.classifier = joblib.load(self.classifier_path)
            print("✨ Both Random Forest Regressor and Classification models loaded successfully!")
        else:
            raise FileNotFoundError(f"Could not find the trained models. Please check:\n1. {self.regressor_path}\n2. {self.classifier_path}")

    def predict(self, user_data):
        """
        user_data: دیکشنری شامل مشخصات کاربر
        """
        # ۱. ترتیب ویژگی‌ها برای مدل رگرسیون (۱۰ ویژگی)
        regressor_features = [
            'Age', 'Is_Male', 'Weight_kg', 'Height_cm', 'BMI', 
            'Body_Fat_Percent', 'Lean_Mass_kg', 'Activity_Score', 
            'Daily_Protein_Intake_g', 'Genetic_Score'
        ]
        
        # ۲. ترتیب ویژگی‌ها برای مدل کلاسیفایر (۹ ویژگی بر اساس فایل pkl ارسالی)
        classifier_features = [
            'Age', 'Gender_Encoded', 'Weight_kg', 'Height_cm', 'BMI', 
            'Body_Fat_Percent', 'Lean_Mass_kg', 'Activity_Score', 
            'Genetic_Score'
        ]
        
        # هماهنگ‌سازی نام متغیر جنسیت برای کلاسیفایر
        if 'Gender_Encoded' not in user_data and 'Is_Male' in user_data:
            user_data['Gender_Encoded'] = user_data['Is_Male']
            
        # تبدیل به آرایه‌های دو بعدی مجزا برای هر مدل
        input_regressor = np.array([[user_data[feat] for feat in regressor_features]])
        input_classifier = np.array([[user_data[feat] for feat in classifier_features]])
        
        # پیش‌بینی مقدار عددی پروتئین مورد نیاز
        protein_pred_raw = self.regressor.predict(input_regressor)
        if isinstance(protein_pred_raw, np.ndarray):
            protein_pred = protein_pred_raw.flatten()[0]
        else:
            protein_pred = protein_pred_raw
            
        # پیش‌بینی نوع مکمل پیشنهادی با ورودی ۹ ویژگی
        supplement_pred = self.classifier.predict(input_classifier)[0]
        
        return {
            "Daily_Protein_Requirement_g": round(float(protein_pred), 2),
            "Recommended_Supplement": supplement_pred
        }

# --- تست پایپ‌لاین اصلاح‌شده در نوت‌بوک ---
if __name__ == "__main__":
    try:
        pipeline = ProteinRecommendationPipeline()
        
        # دیتای تست نمونه
        sample_user = {
            'Age': 28.0,
            'Is_Male': 1,                    # برای رگرسور استفاده می‌شود
            'Weight_kg': 82.5,
            'Height_cm': 180.0,
            'BMI': 25.46,
            'Body_Fat_Percent': 14.5,
            'Lean_Mass_kg': 68.2,
            'Activity_Score': 4.0,           
            'Daily_Protein_Intake_g': 95.0,  
            'Genetic_Score': 50.0            
        }
        
        result = pipeline.predict(sample_user)
        
        print("\n" + "="*40)
        print("🎯 AI RECOMMENDATION SYSTEM OUTPUT:")
        print("="*40)
        print(f"🔹 Daily Protein Target : {result['Daily_Protein_Requirement_g']} grams/day")
        print(f"🔹 Suggested Supplement  : {result['Recommended_Supplement']}")
        print("="*40)
        
    except Exception as e:
        print(f"❌ Error running pipeline: {e}")

✨ Both Random Forest Regressor and Classification models loaded successfully!

🎯 AI RECOMMENDATION SYSTEM OUTPUT:
🔹 Daily Protein Target : 124.24 grams/day
🔹 Suggested Supplement  : Whey_Isolate
